<a href="https://colab.research.google.com/github/AnnaPaula07/RedeNeuralStackOverflow/blob/main/RedeNeuralStackOverflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import confusion_matrix, f1_score, mean_squared_error

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Utilizando dispositivo: {device}')

Utilizando dispositivo: cpu


In [3]:
class MeuDataset(Dataset):

  def __init__(self, X, y):
    self.X = torch.sensor(X, dtype=torch.float32)
    self.y = torch.sensor(y, dtype=torch.float32)

  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

In [16]:
df = pd.read_csv('survey_results_public.csv')
print("Formato original:", df.shape)

df = df[df["AISelect"].notna()].copy()
df["target"] = df["AISelect"].str.startswith("Yes").astype(int)
print("Formato após remover nulos:", df.shape)

features = [
    'Age',
    'EdLevel',
    'Employment',
    'WorkExp',
    'YearsCode',
    'DevType',
    'AISent',
    'AIThreat'
]
target = 'AISelect'

df = df[features + [target]].dropna(subset=[target])
print("Formato após remover colunas:", df.shape)

X = df[features].copy()
y = df[target].copy()

y = y.apply(lambda value: 0 if value.startswith("No") else 1)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

categorical_features = [
    'Age', 'EdLevel', 'Employment', 'DevType', 'AISent', 'AIThreat'
]
numerical_features = [
    'WorkExp', 'YearsCode'
]
categorical_imputer = SimpleImputer(strategy='most_frequent')
numerical_imputer = SimpleImputer(strategy='mean')

X_train[categorical_features] = categorical_imputer.fit_transform(X_train[categorical_features])
X_val[categorical_features] = categorical_imputer.transform(X_val[categorical_features])
X_test[categorical_features] = categorical_imputer.transform(X_test[categorical_features])

X_train[numerical_features] = numerical_imputer.fit_transform(X_train[numerical_features])
X_val[numerical_features] = numerical_imputer.transform(X_val[numerical_features])
X_test[numerical_features] = numerical_imputer.transform(X_test[numerical_features])

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_cat = encoder.fit_transform(X_train[categorical_features])
X_val_cat = encoder.transform(X_val[categorical_features])
X_test_cat = encoder.transform(X_test[categorical_features])

scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train[numerical_features])
X_val_num = scaler.transform(X_val[numerical_features])
X_test_num = scaler.transform(X_test[numerical_features])

X_train = np.hstack([X_train_cat, X_train_num]).astype(np.float32)
X_val = np.hstack([X_val_cat, X_val_num]).astype(np.float32)
X_test = np.hstack([X_test_cat, X_test_num]).astype(np.float32)

y_train = y_train.to_numpy(dtype=np.int64)
y_val = y_val.to_numpy(dtype=np.int64)
y_test = y_test.to_numpy(dtype=np.int64)

print("Treino:", X_train.shape, y_train.shape)
print("Validação:", X_val.shape, y_val.shape)
print("Teste:", X_test.shape, y_test.shape)
print("Classes:", np.unique(y_train))

print("Quantidade de entradas:", X_train.shape[1])

/tmp/ipykernel_1271/409396082.py:1: DtypeWarning: Columns (56,74,92,97,98,105,109,110,132,162,165) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('survey_results_public.csv')


Formato original: (49191, 172)
Formato após remover nulos: (33720, 173)
Formato após remover colunas: (33720, 9)
Treino: (23604, 64) (23604,)
Validação: (5058, 64) (5058,)
Teste: (5058, 64) (5058,)
Classes: [0 1]
Quantidade de entradas: 64
